In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

env = gym.make("CartPole-v1")

class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

obs, _ = env.reset()

for _ in range(1000):
    state = torch.tensor(obs, dtype=torch.float32)
    q_values = model(state)
    action = torch.argmax(q_values).item()

    next_obs, reward, done, truncated, _ = env.step(action)

    target = reward + 0.99 * torch.max(model(torch.tensor(next_obs, dtype=torch.float32)))

    loss = (q_values[action] - target.detach())**2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    obs = next_obs if not done else env.reset()[0]